# Fine-tuning Self-Pretrained ViT on Higher Resolution Images

## Task: Binary Classification (Pizza vs Sushi) from Food-101

This notebook demonstrates how to fine-tune your **own custom pre-trained ViT** (trained on CIFAR-100) for a new task (Food-101 Pizza vs Sushi).

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import time

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # Path to your SAVED model from the previous notebook
    PRETRAINED_PATH = '/content/drive/MyDrive/DL/pre_trained_ViT_CIFAR100.pth'
    DATA_PATH = '/content/drive/MyDrive/DL/Food101'
    print("✅ Connected to Google Drive")
except ImportError:
    print("❌ Not running on Colab, using local path")
    PRETRAINED_PATH = "./saved_models/pre_trained_ViT_CIFAR100.pth" 
    DATA_PATH = "./data"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Mounted at /content/drive
✅ Connected to Google Drive
Using device: cuda


## 1. Define the ViT Architecture
This has to be the exact same class definitions as utilized in the pre-training notebook. Any change here will cause weight loading errors.

In [3]:
class EmbeddedPatches(nn.Module):
    def __init__(self, img_size, in_channels, embed_dim, patch_size, batch_size):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, embed_dim, patch_size, patch_size)
        N = (img_size // patch_size) ** 2
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim))
        self.pos_token = nn.Parameter(torch.randn(1, N+1, embed_dim))

    def forward(self, x):
        B = x.size(0)
        x = self.proj(x)
        x = x.flatten(2).transpose(1, 2)
        cls_token = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_token, x), dim=1)
        x = x + self.pos_token
        return x

class MLP(nn.Module):
    def __init__(self, in_features, out_features, drop_rate):
        super().__init__()
        self.layer1 = nn.Linear(in_features, out_features)
        self.layer2 = nn.Linear(out_features, in_features)
        self.dropout = nn.Dropout(drop_rate)

    def forward(self, x):
        x = self.layer1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.layer2(x)
        x = self.dropout(x)
        return x

class VisionEncoder(nn.Module):
    def __init__(self, embed_dim, msa_size, mlp_dim, enc_dim, drop_rate):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, msa_size, drop_rate, batch_first=True)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = MLP(embed_dim, mlp_dim, drop_rate)

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ViT(nn.Module):
    def __init__(self, img_size, patch_size, batch_size, in_channels, embed_dim, enc_dim, msa_size, mlp_dim, cls_nums, drop_rate):
        super().__init__()
        self.embed = EmbeddedPatches(img_size, in_channels, embed_dim, patch_size, batch_size)
        self.encoder = nn.Sequential(*[
            VisionEncoder(embed_dim, msa_size, mlp_dim, enc_dim, drop_rate)
            for _ in range(enc_dim)
        ])
        self.mlp_head = nn.Linear(embed_dim, cls_nums)

    def forward(self, x):
        x = self.embed(x)
        x = self.encoder(x)
        cls_token = x[:, 0]
        x = self.mlp_head(cls_token)
        return x

## 2. Hyperparameters & Configuration

-   We increase resolution to **64x64**. The model was trained on 32x32.

In [4]:
# Pre-training Config (MUST MATCH THE SAVED MODEL)
OLD_IMG_SIZE = 32
PATCH_SIZE = 4
IN_CHANNELS = 3
EMBED_DIM = 256
MLP_DIM = 512
ENC_NUMS = 6
MSA_NUMS = 8
OLD_CLS_NUMS = 100

# Fine-tuning Config
NEW_IMG_SIZE = 64  # Let's double the resolution to start
NEW_CLS_NUMS = 2   # Pizza vs Sushi
BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-4
DROP_RATE = 0.1

## 3. Data Loading (Food-101 Subset)

In [ ]:
stats = ((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Standard mean/std

train_transform = transforms.Compose([
    transforms.Resize((NEW_IMG_SIZE, NEW_IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

test_transform = transforms.Compose([
    transforms.Resize((NEW_IMG_SIZE, NEW_IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

print("📥 Downloading Food-101...")
full_train_data = datasets.Food101(root=DATA_PATH, split='train', download=True, transform=train_transform)
full_test_data = datasets.Food101(root=DATA_PATH, split='test', download=True, transform=test_transform)

def filter_classes(dataset, target_classes):
    indices = []
    for i, label_idx in enumerate(dataset._labels):
        label_name = dataset.classes[label_idx]
        if label_name in target_classes:
            indices.append(i)
    return indices

target_classes = ['pizza', 'sushi']
train_indices = filter_classes(full_train_data, target_classes)
test_indices = filter_classes(full_test_data, target_classes)

train_subset = Subset(full_train_data, train_indices)
test_dataset = Subset(full_test_data, test_indices)

train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"✅ Filtered Dataset: {len(train_subset)} Train images, {len(test_dataset)} Test images")

📥 Downloading Food-101...


## 4. Interpolation Logic (Custom ViT)

We need to adapt `embed.pos_token` from 32x32 size to 64x64 size.

In [ ]:
def load_custom_vit_weights(model, weights_path, device):
    # 1. Load the saved weights
    state_dict = torch.load(weights_path, map_location=device)
    
    # 2. Get current model's pos_token shape
    new_pos_token = model.embed.pos_token.data
    num_patches_new = new_pos_token.shape[1] - 1
    
    # 3. Get old pos_token from file
    old_pos_token = state_dict['embed.pos_token']
    
    print(f"Interpolating: {old_pos_token.shape} -> {new_pos_token.shape}")

    if old_pos_token.shape != new_pos_token.shape:
        # Extract segments
        old_cls_token = old_pos_token[:, 0:1, :]
        old_grid_tokens = old_pos_token[:, 1:, :]
        
        # Calculate Grid Sizes
        grid_size_old = int(math.sqrt(old_grid_tokens.shape[1])) # Should be 32/4 = 8
        grid_size_new = int(math.sqrt(num_patches_new))          # Should be 64/4 = 16
        
        embed_dim = old_grid_tokens.shape[-1]
        
        # Reshape to (B, Dim, H, W)
        old_grid_tokens = old_grid_tokens.permute(0, 2, 1).reshape(1, embed_dim, grid_size_old, grid_size_old)
        
        # Interpolate
        new_grid_tokens = F.interpolate(
            old_grid_tokens, 
            size=(grid_size_new, grid_size_new), 
            mode='bicubic', 
            align_corners=False
        )
        
        # Reshape back
        new_grid_tokens = new_grid_tokens.flatten(2).transpose(1, 2)
        
        # Recombine
        new_pos_token = torch.cat((old_cls_token, new_grid_tokens), dim=1)
        state_dict['embed.pos_token'] = new_pos_token
        
    # 4. Handle Head Mismatch (100 classes -> 2 classes)
    # Depending on how you saved, it might be 'mlp_head.weight' and 'mlp_head.bias'
    # We usually just DROP the head weights and initialize new ones
    if 'mlp_head.weight' in state_dict:
        del state_dict['mlp_head.weight']
        del state_dict['mlp_head.bias']
        print("Dropped old classification head weights")

    # 5. Load weights (strict=False allows missing head keys)
    model.load_state_dict(state_dict, strict=False)
    return model

In [ ]:
# Initialize Model with NEW Configuration (64x64, 2 Classes)
model = ViT(NEW_IMG_SIZE, PATCH_SIZE, BATCH_SIZE, IN_CHANNELS, 
            EMBED_DIM, ENC_NUMS, MSA_NUMS, MLP_DIM, NEW_CLS_NUMS, DROP_RATE)

model = model.to(device)

# Load and Interpolate weights
model = load_custom_vit_weights(model, PRETRAINED_PATH, device)
print("✅ Custom Model Loaded Successfully")

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/DL/pre_trained_ViT_CIFAR100.pth'

## 5. Fine-Tuning Loop

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

def train(model, loader, optimizer):
    model.train()
    total_loss, correct = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        pizza_idx = full_train_data.class_to_idx['pizza']
        y = (y != pizza_idx).long()

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * x.size(0)
        correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

def validate(model, loader):
    model.eval()
    total_loss, correct = 0, 0
    pizza_idx = full_train_data.class_to_idx['pizza']
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            y = (y != pizza_idx).long()
            out = model(x)
            loss = criterion(out, y)
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
    return total_loss / len(loader.dataset), correct / len(loader.dataset)

print("🚀 Starting Self-Supervised Fine-tuning...")
for epoch in range(EPOCHS):
    start = time.time()
    train_loss, train_acc = train(model, train_loader, optimizer)
    val_loss, val_acc = validate(model, test_loader)
    print(f"Epoch {epoch+1}: Train Acc: {train_acc:.4f}, Test Acc: {val_acc:.4f} | Time: {time.time()-start:.1f}s")